# Test Suite: `EMImputer`
Imports only `EMImputer` and tests imputation accuracy, full-row/column missingness, pandas DataFrames (including nullable dtypes), `transform()` on new data, and scikit-learn compatibility with zero warnings.

In [ ]:
from __future__ import annotations

import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.utils.validation import check_is_fitted

try:
    from imputer import EMImputer
except ImportError:
    from imputer.imputer import EMImputer

## Test 1: Correlated Gaussian Imputation & Accuracy vs. Mean Imputation

In [ ]:
rng = np.random.default_rng(42)
n, d = 300, 4

# Construct a positive-definite covariance matrix with strong feature correlations
A = rng.normal(size=(d, d))
true_cov = A @ A.T + 0.25 * np.eye(d)
true_mean = np.array([5.0, -2.0, 0.0, 12.0])

X_true = rng.multivariate_normal(mean=true_mean, cov=true_cov, size=n)
X_miss = X_true.copy()

miss_mask = rng.random((n, d)) < 0.20
# Include a completely observed row and a completely missing row
miss_mask[0, :] = False
miss_mask[1, :] = True
X_miss[miss_mask] = np.nan

df_miss = pd.DataFrame(X_miss, columns=["x0", "x1", "x2", "x3"])

imp = EMImputer(max_iter=200, tol=1e-7, ridge=1e-6)
df_imp = imp.fit_transform(df_miss)

check_is_fitted(imp)
assert isinstance(df_imp, pd.DataFrame)
assert list(df_imp.columns) == ["x0", "x1", "x2", "x3"]
assert np.isfinite(df_imp.to_numpy()).all()
assert not np.isnan(df_imp.to_numpy()).any()

# Compare RMSE on missing entries against simple column mean imputation
col_means = np.nanmean(X_miss, axis=0)
X_mean_imp = np.where(miss_mask, col_means, X_miss)

rmse_em = np.sqrt(np.mean((X_true[miss_mask] - df_imp.to_numpy()[miss_mask]) ** 2))
rmse_mean = np.sqrt(np.mean((X_true[miss_mask] - X_mean_imp[miss_mask]) ** 2))

print(f"Converged in {imp.n_iter_} iterations | RMSE EM: {rmse_em:.4f} vs Mean: {rmse_mean:.4f}")
assert rmse_em < rmse_mean
print("PASSED Test 1")

## Test 2: `transform()` on Unseen Data & `clone()` / Parameter Checks

In [ ]:
X_unseen = pd.DataFrame(
    {
        "x0": [np.nan, 6.1, np.nan],
        "x1": [-1.5, np.nan, np.nan],
        "x2": [0.4, -0.3, np.nan],
        "x3": [11.8, 12.4, np.nan],
    }
)

out_unseen = imp.transform(X_unseen)
assert isinstance(out_unseen, pd.DataFrame)
assert out_unseen.shape == X_unseen.shape
assert np.isfinite(out_unseen.to_numpy()).all()

# Verify clone() resets fitted state while preserving hyperparameters
cloned = clone(imp)
assert cloned.get_params() == imp.get_params()
assert not hasattr(cloned, "mean_")

print("PASSED Test 2")

## Test 3: Nullable Pandas Dtypes, All-NaN Column, Constant Column & Invalid Hyperparameters

In [ ]:
df_edge = pd.DataFrame(
    {
        "int_col": pd.Series([10, 20, pd.NA, 40, 50], dtype="Int64"),
        "float_col": pd.Series([1.0, pd.NA, 3.0, 4.0, 5.0], dtype="Float64"),
        "const_col": pd.Series([7.0, 7.0, 7.0, 7.0, 7.0], dtype="Float64"),
        "nan_col": pd.Series([pd.NA, pd.NA, pd.NA, pd.NA, pd.NA], dtype="Float64"),
    }
)

imp_edge = EMImputer(max_iter=50, tol=1e-5, ridge=1e-6)
out_edge = imp_edge.fit_transform(df_edge)

assert isinstance(out_edge, pd.DataFrame)
assert out_edge.shape == df_edge.shape
assert np.isfinite(out_edge.to_numpy()).all()
# Constant column should remain at its constant value
assert np.allclose(out_edge["const_col"].to_numpy(), 7.0)

# Invalid parameter checks
for bad_kwargs in [{"max_iter": 0}, {"tol": -1e-4}, {"ridge": -1e-6}]:
    try:
        EMImputer(**bad_kwargs).fit(df_edge)
        raise AssertionError(f"Expected ValueError for {bad_kwargs}")
    except ValueError:
        pass

print("PASSED Test 3")